# Building a Custom BPE Tokenizer Using WikiText-2 Dataset

This notebook demonstrates how to build a Byte Pair Encoding (BPE) tokenizer from scratch using the WikiText-2 dataset from Hugging Face.

## 1. Install Required Libraries

In [ ]:
!pip install datasets tokenizers transformers -q

## 2. Load and Explore the Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import re

# Load WikiText-2 dataset
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1")
print("Dataset structure:")
print(dataset)
print("\nSplits:", list(dataset.keys()))
for split in dataset:
    print(f"  {split}: {len(dataset[split])} rows")

In [ ]:
# Explore sample data
print("Sample training examples:")
for i, example in enumerate(dataset['train'].select(range(10))):
    print(f"[{i}] {repr(example['text'][:100])}")

In [ ]:
# Basic statistics
train_texts = dataset['train']['text']
non_empty = [t for t in train_texts if t.strip()]
print(f"Total training rows: {len(train_texts)}")
print(f"Non-empty rows: {len(non_empty)}")
print(f"Average length of non-empty texts: {sum(len(t) for t in non_empty)/len(non_empty):.1f} chars")
print(f"\nSample text:\n{non_empty[5][:500]}")

## 3. Data Cleaning & Preprocessing

In [ ]:
def clean_text(text):
    """Clean and normalize text."""
    # Remove <unk> tokens
    text = text.replace("<unk>", "")
    # Remove = headers like = = Section = =
    text = re.sub(r'=+\s*([^=]+?)\s*=+', r'\1', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

# Clean all splits
cleaned_train = [clean_text(t) for t in dataset['train']['text']]
cleaned_valid = [clean_text(t) for t in dataset['validation']['text']]
cleaned_test  = [clean_text(t) for t in dataset['test']['text']]

# Remove empty strings
cleaned_train = [t for t in cleaned_train if t]
cleaned_valid = [t for t in cleaned_valid if t]
cleaned_test  = [t for t in cleaned_test  if t]

print(f"After cleaning - Train: {len(cleaned_train)}, Valid: {len(cleaned_valid)}, Test: {len(cleaned_test)}")

In [ ]:
# Deduplication of training text
before_dedup = len(cleaned_train)
cleaned_train_dedup = list(dict.fromkeys(cleaned_train))  # Preserves order, removes exact duplicates
after_dedup = len(cleaned_train_dedup)

print(f"Before deduplication: {before_dedup}")
print(f"After deduplication:  {after_dedup}")
print(f"Removed duplicates:   {before_dedup - after_dedup}")

# Preview cleaned text
print("\nSample cleaned text:")
print(cleaned_train_dedup[10][:300])

## 4. Tokenizer Training

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFD, Lowercase, StripAccents, Sequence as NormSequence
from tokenizers.processors import TemplateProcessing
from tokenizers.decoders import BPEDecoder

# Initialize a BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# Set normalizer: NFD -> Lowercase -> StripAccents
tokenizer.normalizer = NormSequence([NFD(), Lowercase(), StripAccents()])

# Set pre-tokenizer to split on whitespace
tokenizer.pre_tokenizer = Whitespace()

# Set decoder
tokenizer.decoder = BPEDecoder()

# Define special tokens and trainer parameters
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

trainer = BpeTrainer(
    vocab_size=30000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("Tokenizer initialized successfully.")
print(f"Special tokens: {special_tokens}")
print(f"Target vocabulary size: 30,000")

In [ ]:
# Train the tokenizer on deduplicated training data
print("Training BPE tokenizer...")
tokenizer.train_from_iterator(cleaned_train_dedup, trainer=trainer)
print("Training complete!")
print(f"Actual vocabulary size: {tokenizer.get_vocab_size()}")

In [ ]:
# Set post-processor for [CLS] and [SEP] tokens
cls_token_id = tokenizer.token_to_id("[CLS]")
sep_token_id = tokenizer.token_to_id("[SEP]")

tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]", cls_token_id),
        ("[SEP]", sep_token_id),
    ]
)

# Test a quick encoding
sample = "The quick brown fox jumps over the lazy dog."
output = tokenizer.encode(sample)
print(f"Sample: {sample}")
print(f"Tokens: {output.tokens}")
print(f"IDs:    {output.ids}")

## 5. Tokenizer Evaluation

In [ ]:
def compute_metrics(tokenizer, texts, split_name):
    """Compute tokenization metrics for a given text split."""
    total_tokens = 0
    total_chars = 0
    num_sentences = 0
    token_counts = []
    
    for text in texts:
        if not text.strip():
            continue
        encoded = tokenizer.encode(text)
        n_tokens = len(encoded.ids)
        n_chars = len(text)
        total_tokens += n_tokens
        total_chars += n_chars
        token_counts.append(n_tokens)
        num_sentences += 1
    
    avg_tokens = total_tokens / num_sentences if num_sentences > 0 else 0
    compression_ratio = total_chars / total_tokens if total_tokens > 0 else 0
    
    print(f"\n{'='*40}")
    print(f"Metrics for [{split_name}] split")
    print(f"{'='*40}")
    print(f"  Sentences evaluated    : {num_sentences}")
    print(f"  Total tokens           : {total_tokens}")
    print(f"  Total characters       : {total_chars}")
    print(f"  Avg tokens/sentence    : {avg_tokens:.2f}")
    print(f"  Compression ratio      : {compression_ratio:.4f} chars/token")
    
    return {
        "split": split_name,
        "num_sentences": num_sentences,
        "total_tokens": total_tokens,
        "total_chars": total_chars,
        "avg_tokens_per_sentence": round(avg_tokens, 2),
        "compression_ratio": round(compression_ratio, 4)
    }

# Report vocabulary size
vocab_size = tokenizer.get_vocab_size()
print(f"Vocabulary size: {vocab_size}")

# Evaluate on validation and test splits
val_metrics  = compute_metrics(tokenizer, cleaned_valid, "validation")
test_metrics = compute_metrics(tokenizer, cleaned_test,  "test")

In [ ]:
# Tokenization consistency check
print("\nTokenization Consistency Check")
print("="*40)
test_sentences = [
    "The cat sat on the mat.",
    "The cat sat on the mat.",  # Exact duplicate
    "Language models are powerful tools for NLP.",
    "language models are powerful tools for nlp.",  # Lowercase version
]

encodings = [tokenizer.encode(s) for s in test_sentences]
for i, (s, enc) in enumerate(zip(test_sentences, encodings)):
    print(f"\n[{i}] Input : {s}")
    print(f"    Tokens: {enc.tokens}")

# Check consistency: duplicates should yield identical token sequences
assert encodings[0].ids == encodings[1].ids, "Consistency FAILED for exact duplicates!"
assert encodings[2].ids == encodings[3].ids, "Consistency FAILED for case variants (after lowercasing)!"
print("\n✓ Tokenization is consistent (duplicates produce identical encodings).")
print("✓ Case normalization is consistent (lowercase produces same tokens).")

In [ ]:
# Summary table
print("\nSummary")
print("="*50)
summary = pd.DataFrame([val_metrics, test_metrics])
summary.insert(0, 'vocab_size', vocab_size)
print(summary.to_string(index=False))

## 6. Save and Reload the Tokenizer

In [ ]:
import os
from transformers import PreTrainedTokenizerFast

# Save the raw tokenizer JSON
save_dir = "./bpe_tokenizer_wikitext2"
os.makedirs(save_dir, exist_ok=True)

tokenizer.save(os.path.join(save_dir, "tokenizer.json"))
print(f"Raw tokenizer saved to {save_dir}/tokenizer.json")

# Wrap in HuggingFace PreTrainedTokenizerFast for full compatibility
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(save_dir, "tokenizer.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

hf_tokenizer.save_pretrained(save_dir)
print(f"HuggingFace-compatible tokenizer saved to {save_dir}/")
print(f"Files saved: {os.listdir(save_dir)}")

In [ ]:
# Reload the tokenizer and demonstrate encode/decode
print("Reloading tokenizer from disk...")
reloaded_tokenizer = PreTrainedTokenizerFast.from_pretrained(save_dir)
print("Tokenizer reloaded successfully!")
print(f"Vocabulary size: {reloaded_tokenizer.vocab_size}")
print(f"Special tokens: {reloaded_tokenizer.all_special_tokens}")

# Encode example
sample_texts = [
    "Natural language processing enables machines to understand human text.",
    "Wikipedia is a free online encyclopedia that anyone can edit."
]

print("\n--- Encoding Examples ---")
for text in sample_texts:
    encoded = reloaded_tokenizer(text, return_tensors=None)
    token_ids = encoded['input_ids']
    tokens = reloaded_tokenizer.convert_ids_to_tokens(token_ids)
    decoded = reloaded_tokenizer.decode(token_ids, skip_special_tokens=True)
    
    print(f"\nOriginal : {text}")
    print(f"Tokens   : {tokens}")
    print(f"IDs      : {token_ids}")
    print(f"Decoded  : {decoded}")

In [ ]:
# Batch encoding demonstration
print("\n--- Batch Encoding with Padding ---")
batch = reloaded_tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors=None
)

print(f"Input IDs shape: {len(batch['input_ids'])} sequences x {len(batch['input_ids'][0])} tokens")
print(f"Attention masks: {batch['attention_mask']}")

for i, ids in enumerate(batch['input_ids']):
    print(f"\nSequence {i+1} tokens: {reloaded_tokenizer.convert_ids_to_tokens(ids)}")

## Summary

| Step | Description | Status |
|------|-------------|--------|
| Dataset Loading | WikiText-2-v1 from Salesforce/wikitext | ✅ |
| Data Cleaning | Removed `<unk>`, normalized whitespace, deduplication | ✅ |
| BPE Training | Trained with vocab_size=30,000, special tokens | ✅ |
| Evaluation | Vocab size, avg tokens/sentence, compression ratio | ✅ |
| Save & Reload | HuggingFace-compatible format, encode/decode verified | ✅ |